<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/reports/figures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DROID Tech Report — Figure Generation

Generates the 4 qualitative figures for **Section 4.3** of `droid_tech_report.md`.

**Assumes all 3 pipeline stages are already complete on GCS** — no recomputation needed.

| Figure | File | Content |
|--------|------|---------|
| Fig 1 | `fig1_gripper_depth.png` | Raw stereo depth vs refined (SAM + distillation) |
| Fig 2 | `fig2_extrinsics_alignment.png` | Point cloud alignment before/after joint optimization |
| Fig 3 | `fig3_2d_tracking.png` | 2D track overlay on all 3 cameras |
| Fig 4 | `fig4_3d_pointcloud.png` | 3D point cloud colored by env (blue) vs robot (orange) |

All figures are saved to `/content/tech_report_figures/` and bundled as a ZIP for download.

---
## 0. Setup

In [ ]:
import os
import subprocess

REPO_DIR = "/content/droid"
if os.path.exists(REPO_DIR):
  print("⏭️ Already cloned")
  subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)
else:
  subprocess.run(
    ["git", "clone", "--recurse-submodules", "https://github.com/yangyi02/droid.git", REPO_DIR],
    check=True,
  )

os.chdir(REPO_DIR)

In [ ]:
import sys

subprocess.run(
  [sys.executable, "-m", "pip", "install", "-q"]
  + ["mediapy", "plotly", "kaleido", "opencv-python-headless"],
  check=True,
)

In [ ]:
import sys, os, json, glob, zipfile
import numpy as np
import cv2
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import mediapy as media

REPO_DIR = "/content/droid"
if REPO_DIR not in sys.path:
  sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)


FIG_DIR = "/content/tech_report_figures"
os.makedirs(FIG_DIR, exist_ok=True)
print(f"✅ Figures will be saved to: {FIG_DIR}")

In [ ]:
from config import get_config

config = get_config()

episode_id = "ILIAD+5e938e3b+2023-07-20-11h-50m-51s"

GCS_ROOT = "gs://dm-tapnet/tmp/droid"
GCS_DEPTH = f"{GCS_ROOT}/depth"
GCS_EXT = f"{GCS_ROOT}/extrinsics"
GCS_TRACKS = f"{GCS_ROOT}/tracks"

CACHE_DIR = f"/content/droid_cache/{episode_id}"
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"🎯 Episode: {episode_id}")
print(f"📂 Cache:   {CACHE_DIR}")

---
## 1. Load All Pipeline Outputs from GCS

In [ ]:
import json

from config import get_config

config = get_config()

root_path = "/content/droid_raw/1.0.1"
base_url = "https://huggingface.co/KarlP/droid/resolve/main"
files = [
  "camera_serials.json",
  "episode_id_to_path.json",
  "keep_ranges_1_0_1.json",
  "cam2base_extrinsic_superset.json",
]
os.makedirs(root_path, exist_ok=True)
for f in files:
  os.system(f"wget -q -nc -P {root_path} {base_url}/{f}")


def load_json(name):
  with open(os.path.join(root_path, name)) as f:
    return json.load(f)


serials_db = load_json("camera_serials.json")
id_to_path = load_json("episode_id_to_path.json")
keep_ranges = load_json("keep_ranges_1_0_1.json")
extrinsics_db = load_json("cam2base_extrinsic_superset.json")

import core.io
import compute_depth

scene_constants = compute_depth.init_episode(
  episode_id, config.paths.raw, id_to_path, serials_db, keep_ranges
)

camera_ids = list(scene_constants['camera'].keys())
wrist_serial = scene_constants['meta']['wrist_serial']
print(f"✅ Cameras: {camera_ids}")
print(f"✅ Wrist:   {wrist_serial}")

In [ ]:
depth_cache = os.path.join(CACHE_DIR, "depth")
os.makedirs(depth_cache, exist_ok=True)

os.system(f"gsutil cp '{GCS_DEPTH}/{episode_id}/robot.npz' '{depth_cache}/' > /dev/null 2>&1")
robot_data = np.load(os.path.join(depth_cache, "robot.npz"), allow_pickle=True)
for k in ['joint_positions', 'gripper_positions', 'T_cam_ee_init', 'T_ee_base_all']:
  if k in robot_data:
    scene_constants['robot'][k] = robot_data[k]
if 'wrist_serial' in robot_data:
  scene_constants['meta']['wrist_serial'] = str(robot_data['wrist_serial'].item())
  wrist_serial = scene_constants['meta']['wrist_serial']
print("  ✅ robot.npz")

base_files = ["video_left.mp4", "raw_depth.npz", "calibration.npz"]

for cam in camera_ids:
  cam_dir = os.path.join(depth_cache, cam)
  os.makedirs(cam_dir, exist_ok=True)

  cam_files = list(base_files)
  if cam == wrist_serial:
    cam_files += ["original_raw_depth.npz", "gripper_mask.npz"]

  gcs_files = " ".join([f"'{GCS_DEPTH}/{episode_id}/{cam}/{f}'" for f in cam_files])
  os.system(f"gsutil -m cp {gcs_files} '{cam_dir}/' > /dev/null 2>&1")

  vid_path = os.path.join(cam_dir, "video_left.mp4")
  if os.path.exists(vid_path):
    scene_constants['camera'][cam]['video_rgb'] = media.read_video(vid_path)

  d_path = os.path.join(cam_dir, "raw_depth.npz")
  if os.path.exists(d_path):
    scene_constants['camera'][cam]['raw_depth'] = (
      np.load(d_path)['depth'].astype(np.float32) / 1000.0
    )

  c_path = os.path.join(cam_dir, "calibration.npz")
  if os.path.exists(c_path):
    c = np.load(c_path)
    scene_constants['camera'][cam]['K_mat'] = c['K_calib_left']

  if cam == wrist_serial:
    for npz_name, mem_key, is_mm in [
      ("original_raw_depth.npz", "original_raw_depth", True),
      ("gripper_mask.npz", "sam_real_masks", False),
    ]:
      p = os.path.join(cam_dir, npz_name)
      if os.path.exists(p):
        d = np.load(p)
        key = list(d.keys())[0]
        val = d[key].astype(np.float32) / 1000.0 if is_mm else d[key]
        scene_constants['camera'][cam][mem_key] = val

  print(f"  ✅ {cam}")

print("✅ Stage 1 loaded")

In [ ]:
ext_cache = os.path.join(CACHE_DIR, "extrinsics")
os.makedirs(ext_cache, exist_ok=True)

scene_state = {}
for cam in camera_ids:
  cam_dir = os.path.join(ext_cache, cam)
  os.makedirs(cam_dir, exist_ok=True)

  gcs_path = f"{GCS_EXT}/{episode_id}/{cam}/extrinsics.json"
  local_path = os.path.join(cam_dir, "extrinsics.json")
  os.system(f"gsutil cp '{gcs_path}' '{local_path}' > /dev/null 2>&1")

  if os.path.exists(local_path):
    with open(local_path) as f:
      ext_data = json.load(f)
    scene_state[cam] = {
      'base_extrinsic': np.array(ext_data['base_extrinsic'], dtype=np.float32),
      'extrinsics': np.array(ext_data['extrinsics'], dtype=np.float32),
      'is_wrist': ext_data.get('is_wrist', False),
    }
    flag = "🦿" if scene_state[cam]['is_wrist'] else "🎥"
    print(f"  ✅ [{cam}] {flag} {scene_state[cam]['extrinsics'].shape}")
  else:
    print(f"  ⚠️  [{cam}] missing")

print("✅ Stage 2 loaded")

In [ ]:
tracks_cache = os.path.join(CACHE_DIR, "tracks")
os.makedirs(tracks_cache, exist_ok=True)

for fname in ["tracks_3d.npz", "track_metadata.npz"]:
  gcs_p = f"{GCS_TRACKS}/{episode_id}/{fname}"
  local_p = os.path.join(tracks_cache, fname)
  ret = os.system(f"gsutil cp '{gcs_p}' '{local_p}' > /dev/null 2>&1")
  print(f"  {'✅' if ret == 0 else '⚠️ '} {fname}")

data_3d = np.load(os.path.join(tracks_cache, "tracks_3d.npz"))
traj_3d = data_3d["traj_3d"]
vis_global = data_3d["vis_global"]

meta_path = os.path.join(tracks_cache, "track_metadata.npz")
if os.path.exists(meta_path):
  meta = np.load(meta_path)
  n_env = int(meta["n_env"])
  n_robot = int(meta["n_robot"])
  point_type = meta["point_type"]
else:
  n_env, n_robot = traj_3d.shape[1], 0
  point_type = np.zeros(traj_3d.shape[1], dtype=np.uint8)

T, N, _ = traj_3d.shape
print(f"  ✅ tracks_3d: {T} frames × {N} pts ({n_env} env + {n_robot} robot)")

per_cam_tracks = {}
per_cam_vis = {}

for cam in camera_ids:
  cam_dir = os.path.join(tracks_cache, cam)
  os.makedirs(cam_dir, exist_ok=True)
  gcs_cam = f"{GCS_TRACKS}/{episode_id}/{cam}"
  for fname in ["tracks_2d.npz", "intrinsics.npy", "extrinsics_w2c.npy"]:
    os.system(f"gsutil cp '{gcs_cam}/{fname}' '{cam_dir}/' > /dev/null 2>&1")

  t2d = os.path.join(cam_dir, "tracks_2d.npz")
  if os.path.exists(t2d):
    d = np.load(t2d)
    per_cam_tracks[cam] = d["traj_2d"]
    per_cam_vis[cam] = d["vis_2d"]
    print(f"  ✅ {cam}: 2D tracks")
  else:
    print(f"  ⚠️  {cam}: tracks_2d.npz missing")

print("✅ Stage 3 loaded")

---
## 2. Generate Figures

In [ ]:
wrist = scene_constants['camera'][wrist_serial]

if 'original_raw_depth' not in wrist:
  print("⚠️ original_raw_depth not in GCS — this needs the wrist-camera extra files.")
else:
  gripper_pos = scene_constants['robot']['gripper_positions']
  frame_idx = int(np.argmin(gripper_pos))

  rgb = wrist['video_rgb'][frame_idx]
  depth_raw = wrist['original_raw_depth'][frame_idx]
  depth_refine = wrist['raw_depth'][frame_idx]
  gripper_mask = wrist.get('sam_real_masks', [None])[frame_idx]

  def depth_colormap(d, vmax=0.5):
    valid = d > 0
    norm = np.zeros_like(d)
    norm[valid] = np.clip(d[valid], 0, vmax) / vmax * 255
    colored = cv2.applyColorMap(norm.astype(np.uint8), cv2.COLORMAP_MAGMA)
    colored = cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)
    colored[~valid] = 0
    return colored

  rgb_display = rgb.copy()
  if gripper_mask is not None:
    contours, _ = cv2.findContours(
      gripper_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    cv2.drawContours(rgb_display, contours, -1, (0, 255, 0), 3)

  fig, axes = plt.subplots(1, 3, figsize=(18, 5))
  fig.suptitle(
    f"Gripper Depth Refinement  •  Frame {frame_idx}  •  Wrist [{wrist_serial}]",
    fontsize=13,
    fontweight='bold',
  )

  axes[0].imshow(rgb_display)
  axes[0].set_title("RGB  +  SAM gripper mask (green outline)", fontsize=11)

  axes[1].imshow(depth_colormap(depth_raw))
  axes[1].set_title("Raw stereo depth\n(gripper: noisy / missing)", fontsize=11)

  axes[2].imshow(depth_colormap(depth_refine))
  axes[2].set_title("Refined depth\n(distilled from open-gripper frames)", fontsize=11)

  for ax in axes:
    ax.axis('off')

  plt.tight_layout()
  save_path = os.path.join(FIG_DIR, "fig1_gripper_depth.png")
  plt.savefig(save_path, dpi=150, bbox_inches='tight')
  plt.show()
  print(f"✅ {save_path}")

In [ ]:
FRAME = 0
VMAX = 2.0

cam_colors = [np.array([160, 30, 30]), np.array([30, 140, 30]), np.array([30, 30, 160])]


def unproject(depth, K, extr_c2w):
  H, W = depth.shape
  u, v = np.meshgrid(np.arange(W), np.arange(H))
  z = depth.ravel()
  valid = z > 0.05
  if K.shape == (3, 3):
    fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]
  else:
    fx, fy, cx, cy = K
  x = (u.ravel() - cx) / fx * z
  y = (v.ravel() - cy) / fy * z
  pts_cam = np.stack([x, y, z, np.ones_like(z)], axis=1)
  pts_w = (extr_c2w @ pts_cam[valid].T).T[:, :3]
  return pts_w


def scatter_xy(ax, pts, color_rgb, title, xlim=(-1.5, 1.5), ylim=(-0.5, 3.0), n=25000):
  idx = np.random.permutation(len(pts))[:n]
  ax.scatter(pts[idx, 0], pts[idx, 1], c=[color_rgb / 255], s=0.4, alpha=0.5)
  ax.set_xlim(xlim)
  ax.set_ylim(ylim)
  ax.set_aspect('equal')
  ax.set_title(title, fontsize=11)
  ax.set_xlabel('X (m)')
  ax.set_ylabel('Y (m)')


fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Extrinsics Calibration  •  Top-down view  •  Frame 0", fontsize=13, fontweight='bold')

all_pts_before, all_pts_after = [], []

for i, cam in enumerate(camera_ids):
  depth = scene_constants['camera'][cam]['raw_depth'][FRAME]
  K = scene_constants['camera'][cam]['K_mat']
  state = scene_state[cam]

  base_w2c = state['base_extrinsic']
  opt_w2c = state['extrinsics'][FRAME]
  base_c2w = np.linalg.inv(base_w2c)
  opt_c2w = np.linalg.inv(opt_w2c)

  pts_before = unproject(depth, K, base_c2w)
  pts_after = unproject(depth, K, opt_c2w)

  col = cam_colors[i % len(cam_colors)]
  scatter_xy(axes[0, i], pts_before, col, f"BEFORE  [{cam[:8]}]\n(VGGT init only)")
  scatter_xy(axes[1, i], pts_after, col, f"AFTER  [{cam[:8]}]\n(Chamfer + robot)")

  all_pts_before.append((pts_before, col))
  all_pts_after.append((pts_after, col))

plt.tight_layout()
save_path = os.path.join(FIG_DIR, "fig2_extrinsics_alignment.png")
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ {save_path}")

In [ ]:
import core.visualization

mid = T // 2
TAIL = 20

colors = np.zeros((N, 3), dtype=np.float32)
env_m = point_type == 0
robot_m = point_type == 1

if env_m.any():
  y_vals = traj_3d[mid, env_m, 1]
  norm = plt.Normalize(np.nanmin(y_vals), np.nanmax(y_vals))
  colors[env_m] = plt.cm.gist_rainbow(norm(y_vals))[:, :3] * 255
colors[robot_m] = [255, 140, 0]

fig, axes = plt.subplots(1, len(camera_ids), figsize=(7 * len(camera_ids), 5))
if len(camera_ids) == 1:
  axes = [axes]
fig.suptitle(
  f"2D Track Overlay  •  Frame {mid}/{T}  •  env=rainbow, robot=orange",
  fontsize=13,
  fontweight='bold',
)

t0 = max(0, mid - TAIL)

for i, cam in enumerate(camera_ids):
  if cam not in per_cam_tracks:
    axes[i].set_title(f"{cam}\n(no tracks)")
    axes[i].axis('off')
    continue

  rgb_frames = scene_constants['camera'][cam]['video_rgb'][t0 : mid + 1]
  trk_slice = per_cam_tracks[cam][t0 : mid + 1]
  vis_slice = per_cam_vis[cam][t0 : mid + 1]

  rendered = core.visualization.render_2d_tracking_video(
    list(rgb_frames),
    trk_slice,
    vis_slice,
    global_colors=colors,
    linewidth=2,
    tracks_leave_trace=TAIL,
  )

  label = "Wrist" if cam == wrist_serial else "External"
  axes[i].imshow(rendered[-1])
  axes[i].set_title(f"{label}\n[{cam}]", fontsize=11)
  axes[i].axis('off')

plt.tight_layout()
save_path = os.path.join(FIG_DIR, "fig3_2d_tracking.png")
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ {save_path}")

In [ ]:
mid = T // 2
pts = traj_3d[mid]
vis = vis_global[mid]

env_m = (point_type == 0) & vis
robot_m = (point_type == 1) & vis

MAX = 3000
env_pts = pts[env_m][np.random.permutation(env_m.sum())[:MAX]]
robot_pts = pts[robot_m][np.random.permutation(robot_m.sum())[:MAX]]

fig = go.Figure(
  [
    go.Scatter3d(
      x=env_pts[:, 0],
      y=env_pts[:, 1],
      z=env_pts[:, 2],
      mode='markers',
      marker=dict(size=2, color='steelblue', opacity=0.7),
      name=f'Environment ({env_m.sum()} pts)',
    ),
    go.Scatter3d(
      x=robot_pts[:, 0],
      y=robot_pts[:, 1],
      z=robot_pts[:, 2],
      mode='markers',
      marker=dict(size=3, color='orangered', opacity=0.9),
      name=f'Robot ({robot_m.sum()} pts)',
    ),
  ]
)
fig.update_layout(
  title=f"3D Point Cloud  •  Frame {mid}/{T}<br><sub>Blue = Environment | Orange = Robot</sub>",
  height=650,
  margin=dict(l=0, r=0, b=0, t=60),
  scene=dict(
    aspectmode='data',
    camera=dict(eye=dict(x=-1.2, y=-1.2, z=0.8)),
    xaxis_title='X (m)',
    yaxis_title='Y (m)',
    zaxis_title='Z (m)',
  ),
  legend=dict(x=0.01, y=0.99),
)
fig.show(renderer='colab')

save_path = os.path.join(FIG_DIR, "fig4_3d_pointcloud.png")
fig.write_image(save_path, width=1200, height=700, scale=2)
print(f"✅ {save_path}")

---
## 3. Download / Upload Figures

In [ ]:
from google.colab import files

figs = sorted(
  glob.glob(os.path.join(FIG_DIR, "*.png")) + glob.glob(os.path.join(FIG_DIR, "*.html"))
)

print("Generated files:")
for f in figs:
  print(f"  {os.path.basename(f):45s}  {os.path.getsize(f) / 1024:.0f} KB")

zip_path = "/content/tech_report_figures.zip"
with zipfile.ZipFile(zip_path, 'w') as zf:
  for f in figs:
    zf.write(f, os.path.basename(f))
print(f"\n📦 ZIP ready: {zip_path}")


files.download(zip_path)